In [2]:
# Cell 1: get the code and the training stack
!git clone https://github.com/SachiK0thari/care-nudge.git
%cd care-nudge
!pip install -q -r requirements-dev.txt

Cloning into 'care-nudge'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 31 (delta 1), reused 31 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 28.00 KiB | 4.00 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/care-nudge
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
# Cell 2: log in to Hugging Face (paste a write token)
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [ ]:
# Cell 3: data + two baselines from the un-tuned base model: the same prompt, then the prompt plus worked examples.
!python scripts/build_dataset.py
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --name base
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --few-shot --name base_fewshot

val       63 -> data/val.jsonl
train    570 -> data/train.jsonl
test     260 -> data/test.jsonl
probes    12 -> data/probes.jsonl
train rows per level: ROUTINE 96, SOON 132, URGENT 130, EMERGENCY 212
distinct replies: 30 across 570 rows (most repeated: 24x)
config.json: 100% 727/727 [00:00<00:00, 2.28MB/s]
tokenizer_config.json: 100% 9.38k/9.38k [00:00<00:00, 2.49MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 10.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 15.3MB/s]

tokenizer.json: downloading bytes:   9% 986k/11.4M [00:00<00:04, 2.22MB/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 5.66MB/s,  335kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 19.0MB/s, 1.13MB/s  ]
model.safetensors.index.json: 100% 32.8k/32.8k [00:00<00:00, 42.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.99G [00:00<?, ?B/s]    

In [ ]:
!nvidia-smi
import torch; print(torch.cuda.is_available())

In [ ]:
# Cell 4: fine-tune, then evaluate the adapter on the test split and on the probe set
!python scripts/train.py --push-to SachiK/carenudge-lora
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --name finetuned
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --test data/probes.jsonl --name finetuned_probes

Generating train split: 570 examples [00:00, 33704.39 examples/s]
Generating val split: 63 examples [00:00, 11339.36 examples/s]
Loading weights: 100% 398/398 [03:25<00:00,  1.93it/s]
generation_config.json: 100% 238/238 [00:00<00:00, 705kB/s]
Tokenizing train dataset: 100% 570/570 [00:00<00:00, 648.74 examples/s]
Building labels for train dataset: 100% 570/570 [00:00<00:00, 2957.59 examples/s]
Truncating train dataset: 100% 570/570 [00:00<00:00, 5409.90 examples/s]
Dropping fully masked examples from train dataset: 100% 570/570 [00:00<00:00, 21265.13 examples/s]
Tokenizing eval dataset: 100% 63/63 [00:00<00:00, 393.85 examples/s]
Building labels for eval dataset: 100% 63/63 [00:00<00:00, 1612.77 examples/s]
Truncating eval dataset: 100% 63/63 [00:00<00:00, 2085.88 examples/s]
Dropping fully masked examples from eval dataset: 100% 63/63 [00:00<00:00, 6190.49 examples/s]
  0% 0/108 [00:00<?, ?it/s]

In [1]:
!nvidia-smi
import torch; print(torch.cuda.is_available())

Tue Nov 17 00:50:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 5: merge the adapter and publish the model the Space will load
!python scripts/merge.py --push-to SachiK/carenudge-qwen3-4b